In [1]:
import requests

#Pharmacy list from NHS
url = "https://opendata.nhsbsa.net/dataset/240d142d-df82-4e97-b051-12371519e4e1/resource/7bd7b192-d3a5-44f2-8f7d-3c510cc4a8e5/download/consol_pharmacy_list_202526q3final.csv"

out_path = "consol_pharmacy_list.csv"

resp = requests.get(url, timeout=30)
resp.raise_for_status()  # raise if 4xx/5xx
with open(out_path, "wb") as f:
    f.write(resp.content)

print(f"Saved to {out_path}")

Saved to consol_pharmacy_list.csv


In [4]:
import os
import pandas as pd

#below is trying to use spark to ingest data
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, lower, regexp_extract
from pyspark import SparkContext

spark = SparkSession.builder.getOrCreate()
df_pyspark = spark.read.option('header','true').csv('consol_pharmacy_list.csv',inferSchema=True)


In [8]:
df_pyspark.columns

['PHARMACY_ODS_CODE_F_CODE',
 'HEALTH_AND_WELLBEING_BOARD',
 'PHARMACY_TRADING_NAME',
 'ORGANISATION_NAME',
 'ADDRESS_FIELD_1',
 'ADDRESS_FIELD_2',
 'ADDRESS_FIELD_3',
 'ADDRESS_FIELD_4',
 'POST_CODE',
 'PHARMACY_OPENING_HOURS_MONDAY',
 'MON_TOTAL',
 'PHARMACY_OPENING_HOURS_TUESDAY',
 'TUES_TOTAL',
 'PHARMACY_OPENING_HOURS_WEDNESDAY',
 'WED_TOTAL',
 'PHARMACY_OPENING_HOURS_THURSDAY',
 'THURS_TOTAL',
 'PHARMACY_OPENING_HOURS_FRIDAY',
 'FRI_TOTAL',
 'PHARMACY_OPENING_HOURS_SATURDAY',
 'SAT_TOTAL',
 'PHARMACY_OPENING_HOURS_SUNDAY',
 'SUN_TOTAL',
 'WEEKLY_TOTAL',
 'CONTRACT_TYPE']

In [9]:
phlist = df_pyspark.toPandas()

In [15]:
## Export the Adress postcode lib to SQL Server

from sqlalchemy import create_engine
from sqlalchemy.types import Integer, String, Numeric, Float

server = 'cic-dw02'
database = 'cicedm01_dev'
engine = create_engine(
    'mssql+pyodbc://'+server+'/'+database+'?driver=ODBC+Driver+17+for+SQL+Server',fast_executemany=True
    # Or ODBC Driver 18
)

dtype_map = {
    "PHARMACY_ODS_CODE_F_CODE": String(5), 
    "HEALTH_AND_WELLBEING_BOARD": String(100),
    "PHARMACY_TRADING_NAME": String(100),
    "ORGANISATION_NAME": String(100),
    "ADDRESS_FIELD_1": String(250),
    "ADDRESS_FIELD_2": String(250),
    "ADDRESS_FIELD_3": String(250),
    "ADDRESS_FIELD_4": String(250),
    "POST_CODE": String(10),
    "PHARMACY_OPENING_HOURS_MONDAY": String(100),
    "MON_TOTAL": Float,
    "PHARMACY_OPENING_HOURS_TUESDAY": String(100),
    "TUES_TOTAL": Float,
    "PHARMACY_OPENING_HOURS_WEDNESDAY": String(100),
    "WED_TOTAL": Float,
    "PHARMACY_OPENING_HOURS_THURSDAY": String(100),
    "THURS_TOTAL": Float,
    "PHARMACY_OPENING_HOURS_FRIDAY": String(100),
    "FRI_TOTAL": Float,
    "PHARMACY_OPENING_HOURS_SATURDAY": String(100),
    "SAT_TOTAL": Float,
    "PHARMACY_OPENING_HOURS_SUNDAY": String(100),
    "SUN_TOTAL": Float,
    "WEEKLY_TOTAL": Float,
    "CONTRACT_TYPE": String(100)
}

phlist.to_sql(
    name="PharmacyListFromNHS",
    con=engine,
    schema="Landing",
    if_exists="replace",
    index=False,
    chunksize = 5000,
    dtype=dtype_map
    #method="multi"   # batch insert
)

-3